In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

file_path = "/content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Premier League/premier_league.csv"
df = pd.read_csv(file_path)
df.sample(n=20)

,season,date,home_team,away_team,hometeamgoals,awayteamgoals,hometeamresult,home_team_points,away_team_points,OddHome,OddDraw,OddAway
2259,2019.0,2019-08-23,Aston Villa,Everton,2.0,0.0,win,3.0,0.0,3.25,3.50,2.15
7682,2002.0,2003-03-16,West Brom,Chelsea,0.0,2.0,loss,0.0,3.0,4.33,3.25,1.72
4699,2012.0,2013-02-02,Wigan,Southampton,2.0,2.0,draw,1.0,1.0,2.20,3.50,3.50
5935,2009.0,2009-12-09,Stoke,Chelsea,1.0,2.0,loss,0.0,3.0,9.00,4.50,1.40
6864,2006.0,2007-04-28,Middlesbrough,Tottenham,2.0,3.0,loss,0.0,3.0,2.70,3.25,2.50
4422,2013.0,2013-12-01,Reading,West Brom,3.0,2.0,win,3.0,0.0,3.00,3.40,2.50
2205,2019.0,2019-10-05,Burnley,Everton,1.0,0.0,win,3.0,0.0,2.87,3.30,2.50
446,2023.0,2024-04-13,Bournemouth,Man United,2.0,2.0,draw,1.0,1.0,2.40,3.90,2.60
7547,2005.0,2005-08-23,Birmingham,Middlesbrough,0.0,3.0,loss,0.0,3.0,2.50,3.20,2.75
2791,2017.0,2018-01-31,Everton,Leicester,2.0,1.0,win,3.0,0.0,3.00,3.20,2.62


In [12]:
# --- Ensure types ---

df["season"] = pd.to_numeric(df["season"], errors="coerce").astype("Int64")
for c in ["hometeamgoals","awayteamgoals","home_team_points","away_team_points"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

use = df.dropna(subset=[
    "season","home_team","away_team",
    "hometeamgoals","awayteamgoals",
    "home_team_points","away_team_points","hometeamresult"
]).copy()

In [13]:
# --- Building per-team rows (home & away) for ALL seasons ---

home = pd.DataFrame({
    "season": use["season"].astype(int),
    "team": use["home_team"],
    "points": use["home_team_points"].astype(int),
    "wins":  (use["hometeamresult"] == "win").astype(int),
    "draws": (use["hometeamresult"] == "draw").astype(int),
    "losses":(use["hometeamresult"] == "loss").astype(int),
    "gf": use["hometeamgoals"].astype(int),
    "ga": use["awayteamgoals"].astype(int),
})

away = pd.DataFrame({
    "season": use["season"].astype(int),
    "team": use["away_team"],
    "points": use["away_team_points"].astype(int),
    "wins":  (use["hometeamresult"] == "loss").astype(int),   # away win when home loses
    "draws": (use["hometeamresult"] == "draw").astype(int),
    "losses":(use["hometeamresult"] == "win").astype(int),
    "gf": use["awayteamgoals"].astype(int),
    "ga": use["hometeamgoals"].astype(int),
})

long = pd.concat([home, away], ignore_index=True)

In [14]:
# --- Combined to end-of-season table for each (season, team) ---

agg = (long
       .groupby(["season","team"], as_index=False)
       .sum(numeric_only=True))
agg["gd"] = agg["gf"] - agg["ga"]


In [15]:
# --- Rank within each season: Points ↓, GD ↓, GF ↓, Team ↑ ---

agg = agg.sort_values(
    ["season","points","gd","gf","team"],
    ascending=[True, False, False, False, True],
    kind="mergesort"
)
agg["rank"] = agg.groupby("season").cumcount() + 1


In [16]:
# --- Final view (only required columns) ---

standings_all = (
    agg.rename(columns={"team":"team_name"})
       [["season","team_name","points","wins","draws","losses","rank"]]
       .sort_values(["season","rank"], ascending=[True, True])
       .reset_index(drop=True)
)


In [17]:
print("Seasons covered:", standings_all["season"].min(), "→", standings_all["season"].max())
print("Rows:", len(standings_all))
display(standings_all.head(40))

Seasons covered: 2002 → 2024
Rows: 494


,season,team_name,points,wins,draws,losses,rank
0,2002,Man United,83,25,8,5,1
1,2002,Arsenal,78,23,9,6,2
2,2002,Newcastle,69,21,6,11,3
3,2002,Chelsea,67,19,10,9,4
4,2002,Liverpool,64,18,10,10,5
5,2002,Blackburn,60,16,12,10,6
6,2002,Everton,59,17,8,13,7
7,2002,Southampton,52,13,13,12,8
8,2002,Man City,51,15,6,17,9
9,2002,Tottenham,50,14,8,16,10


In [18]:
out_path = "/content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Premier League/pl_standings_all_seasons.csv"
standings_all.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /content/drive/MyDrive/Sophomore Year/IML_Fall2025_SkillVersusLuck/Premier League/pl_standings_all_seasons.csv
